# Вебинар 7: NLP — Большие языковые модели (LLM) в финтехе

## 📊 Датасет 1: Financial Phrase Bank
**Название:** [Financial Phrase Bank](https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news)  
**Описание:** 4840+ финансовых новостей с экспертной разметкой тональности.  
**Задача:** Классификация финансовых текстов с использованием предобученных трансформеров.

## 📊 Датасет 2: NER
**Название:** Financial NER Dataset — для извлечения сущностей.  
**Задача:** Извлечение финансовых сущностей (суммы, компании, даты).

## 🎯 Цели ноутбука
1. Классификация тональности с предобученным DistilBERT
2. NER для извлечения финансовых сущностей
3. Генерация финансовых отчётов
4. Суммаризация длинных текстов

## 1. Импорт и загрузка

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import os
import glob
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
files = glob.glob('/kaggle/input/**/all-data.csv', recursive=True)
if not files:
    files = glob.glob('/kaggle/input/**/FinancialPhraseBank*.csv', recursive=True)

if files:
    df = pd.read_csv(files[0], encoding='latin-1', names=['sentiment', 'text'])
else:
    np.random.seed(42)
    data = []
    
    positive = [
        'Company reports record profits in the third quarter',
        'Stock surges on strong earnings beat',
        'Analysts upgrade to buy rating citing growth',
        'Dividend increase announced by major bank',
        'Revenue grew 20 percent year over year',
    ]
    negative = [
        'Shares plunge after disappointing results',
        'Company misses earnings estimates significantly',
        'Major lawsuit filed against financial firm',
        'Credit rating downgraded to junk status',
        'Profit warning sends stock to 52-week low',
    ]
    neutral = [
        'Company to release earnings next week',
        'Board meeting scheduled for next month',
        'Annual report filed with regulators',
        'New CFO appointed effective immediately',
        'Company updates investor presentation',
    ]
    
    for template, sent in [(positive, 'positive'), (negative, 'negative'), (neutral, 'neutral')]:
        for _ in range(300):
            data.append({'sentiment': sent, 'text': np.random.choice(template)})
    
    df = pd.DataFrame(data)

print(f"Dataset shape: {df.shape}")
print(df['sentiment'].value_counts())
print()
print(df.head())

## 2. TF-IDF Baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

X = df['text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred = lr.predict(X_test_tfidf)
baseline_acc = accuracy_score(y_test, y_pred)
print(f"Baseline (TF-IDF + LR) Accuracy: {baseline_acc:.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
# Визуализация confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['negative', 'neutral', 'positive'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'])
plt.title('Confusion Matrix (TF-IDF + LR)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

## 3. DistilBERT для классификации

In [ ]:
try:
    import torch
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        Trainer, TrainingArguments
    )
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    TRANSFORMERS_AVAILABLE = True
except ImportError as e:
    print(f"Transformers not installed: {e}")
    print("Install: pip install transformers torch")
    TRANSFORMERS_AVAILABLE = False

In [ ]:
if TRANSFORMERS_AVAILABLE:
    label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
    id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
    
    df['label'] = df['sentiment'].map(label_map)
    
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['text'].values, df['label'].values, test_size=0.2, random_state=42, stratify=df['label']
    )
    
    MODEL_NAME = 'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
    test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)
    
    print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")
else:
    print("Skip DistilBERT section - install transformers")

In [ ]:
if TRANSFORMERS_AVAILABLE:
    from torch.utils.data import Dataset
    
    class FinancialDataset(Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
        
        def __len__(self):
            return len(self.labels)
        
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item
    
    train_dataset = FinancialDataset(train_encodings, train_labels)
    test_dataset = FinancialDataset(test_encodings, test_labels)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=id2label, label2id=label_map
    )
    
    print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
if TRANSFORMERS_AVAILABLE:
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=50,
        evaluation_strategy='epoch',
        save_strategy='no',
        report_to='none',
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
    )
    
    trainer.train()
    
    predictions = trainer.predict(test_dataset)
    y_pred_bert = np.argmax(predictions.predictions, axis=1)
    
    bert_acc = accuracy_score(test_labels, y_pred_bert)
    print(f"\nDistilBERT Accuracy: {bert_acc:.4f}")
    print(f"Baseline (TF-IDF):    {baseline_acc:.4f}")
    print(f"Improvement: {bert_acc - baseline_acc:+.4f}")
    print(classification_report(test_labels, y_pred_bert, target_names=['negative', 'neutral', 'positive']))

## 4. NER: Извлечение финансовых сущностей

In [ ]:
import re

def extract_financial_entities(text):
    """Извлечение финансовых сущностей через regex"""
    entities = {}
    
    # Money: $1.5 billion, 50 million, etc.
    money_patterns = [
        r'\$\s*\d+(?:\.\d+)?\s*(?:billion|million|thousand|bn|mn|k)?',
        r'\d+(?:\.\d+)?\s*(?:billion|million|thousand|bn|mn|k)\s*(?:dollars|USD)?',
        r'\d+(?:\.\d+)?%',
    ]
    amounts = []
    for pattern in money_patterns:
        amounts.extend(re.findall(pattern, text, re.IGNORECASE))
    if amounts:
        entities['amounts'] = amounts
    
    # Companies (capitalized words)
    company_pattern = r'\b[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*\s+(?:Inc|Corp|Ltd|LLC|Company|Group|Holdings|Bank|Industries)\b'
    companies = re.findall(company_pattern, text)
    if companies:
        entities['companies'] = list(set(companies))
    
    # Dates
    date_pattern = r'\b(?:Q[1-4]|quarter|annual|year|fiscal)\s+\d{4}\b|\b\d{4}\b'
    dates = re.findall(date_pattern, text, re.IGNORECASE)
    if dates:
        entities['periods'] = list(set(dates))
    
    # Financial metrics
    metrics = ['revenue', 'profit', 'earnings', 'EBITDA', 'EPS', 'dividend', 'income', 'loss']
    found_metrics = [m for m in metrics if m.lower() in text.lower()]
    if found_metrics:
        entities['metrics'] = found_metrics
    
    return entities

In [ ]:
# Тест NER
test_texts = [
    "Apple Inc reported Q3 2024 revenue of $89.5 billion, beating analyst expectations by 5%.",
    "Goldman Sachs Group announced a dividend increase of 15%, reaching $3.00 per share.",
    "Tesla Inc shares fell 8% after the company missed its 2024 earnings estimates.",
    "JPMorgan Chase & Co reported record profits of $50.3 billion for fiscal year 2023.",
    "Microsoft Corporation's quarterly earnings showed 12% growth in cloud services revenue.",
]

print("NER Results:\n")
for text in test_texts:
    print(f"Text: {text}")
    entities = extract_financial_entities(text)
    for entity_type, values in entities.items():
        print(f"  {entity_type}: {values}")
    print()

In [ ]:
# === NER с использованием предобученной модели (если доступна) ===
if TRANSFORMERS_AVAILABLE:
    try:
        from transformers import pipeline
        
        # Используем BERT для NER
        ner_pipeline = pipeline(
            'ner', 
            model='dslim/bert-base-NER',
            aggregation_strategy='simple'
        )
        
        print("BERT NER Pipeline loaded\n")
        
        for text in test_texts[:2]:
            print(f"Text: {text}")
            entities = ner_pipeline(text)
            for e in entities:
                print(f"  {e['entity_group']}: {e['word']} (score: {e['score']:.3f})")
            print()
    except Exception as e:
        print(f"BERT NER not available: {e}")

## 5. Генерация финансового отчёта

In [ ]:
def generate_financial_report(client_data):
    name = client_data['name']
    income = client_data['income']
    expenses = client_data['expenses']
    savings = client_data['savings']
    investments = client_data['investments']
    debts = client_data['debts']
    credit_score = client_data['credit_score']
    
    savings_rate = (income - expenses) / income * 100 if income > 0 else 0
    debt_to_income = debts / income * 100 if income > 0 else 0
    emergency_months = savings / expenses if expenses > 0 else 0
    net_worth = savings + investments - debts
    
    if credit_score >= 750:
        score_label = 'Excellent'
    elif credit_score >= 650:
        score_label = 'Good'
    elif credit_score >= 550:
        score_label = 'Fair'
    else:
        score_label = 'Poor'
    
    report = f"""
=== PERSONAL FINANCIAL REPORT ===
Client: {name}
Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}

--- INCOME & EXPENSES ---
Monthly Income: {income:,.0f} RUB
Monthly Expenses: {expenses:,.0f} RUB
Savings Rate: {savings_rate:.1f}%

--- ASSETS ---
Savings: {savings:,.0f} RUB
Investments: {investments:,.0f} RUB
Total Assets: {savings + investments:,.0f} RUB

--- LIABILITIES ---
Debts: {debts:,.0f} RUB
Debt-to-Income: {debt_to_income:.1f}%

--- NET WORTH ---
Net Worth: {net_worth:,.0f} RUB

--- CREDIT SCORE ---
Score: {credit_score}/850 ({score_label})

--- RECOMMENDATIONS ---
"""
    
    recommendations = []
    if savings_rate < 10:
        recommendations.append(f"Increase savings rate. Current {savings_rate:.1f}% is below recommended 20%.")
    elif savings_rate >= 20:
        recommendations.append(f"Excellent savings rate ({savings_rate:.1f}%). Consider increasing investment allocation.")
    
    if emergency_months < 3:
        recommendations.append(f"Emergency fund covers only {emergency_months:.1f} months. Recommended: 3-6 months.")
    elif emergency_months >= 6:
        recommendations.append(f"Emergency fund is sufficient ({emergency_months:.0f} months). Excess can be invested.")
    
    if debt_to_income > 40:
        recommendations.append(f"High debt burden ({debt_to_income:.1f}%). Prioritize debt repayment.")
    
    if credit_score < 650:
        recommendations.append(f"Credit score {credit_score} is below average. Reduce credit utilization, avoid late payments.")
    
    if not recommendations:
        recommendations.append("Financial situation is stable. Continue current strategy.")
    
    report += "\n".join([f"- {r}" for r in recommendations])
    report += "\n\n=== END OF REPORT ==="
    
    return report

In [ ]:
# Тест генератора отчётов
client = {
    'name': 'Ivan Petrov',
    'income': 150000,
    'expenses': 120000,
    'savings': 200000,
    'investments': 50000,
    'debts': 300000,
    'credit_score': 680,
}

report = generate_financial_report(client)
print(report)

## 6. Суммаризация финансовых отчётов

In [ ]:
from collections import Counter

def extractive_summarize(text, n_sentences=3):
    """Упрощённая экстрактивная суммаризация"""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    
    if len(sentences) <= n_sentences:
        return text
    
    words_per_sentence = [s.lower().split() for s in sentences]
    all_words = [w for sent in words_per_sentence for w in sent]
    word_freq = Counter(all_words)
    
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'was', 'were', 'are', 'been'}
    
    sentence_scores = []
    for i, words in enumerate(words_per_sentence):
        score = sum(word_freq.get(w, 0) for w in words if w not in stop_words)
        position_bonus = 1.0 / (1 + 0.1 * i)
        number_bonus = 1.5 if any(re.search(r'\d', w) for w in words) else 1.0
        sentence_scores.append((i, score * position_bonus * number_bonus))
    
    sentence_scores.sort(key=lambda x: -x[1])
    top_indices = sorted([idx for idx, _ in sentence_scores[:n_sentences]])
    
    summary = '. '.join(sentences[i] for i in top_indices) + '.'
    return summary

In [ ]:
# Длинный финансовый отчёт
long_report = """
Sberbank published its financial results for the third quarter of 2024.
Net profit reached 350 billion rubles, which is 15% higher than the same period last year.
The growth was driven by a 12% increase in the loan portfolio and a reduction in cost of risk.
Retail lending grew by 18% due to high demand for mortgages and consumer loans.
Corporate portfolio increased by 8%, with oil and gas companies remaining the main borrowers.
Commission income grew by 22% thanks to ecosystem development and digital services.
Sberbank's mobile app reached 45 million active users.
Return on equity (ROE) was 24%, one of the best in the industry.
The board of directors recommended dividends of 35 rubles per share.
Analysts positively evaluated the results, with the average target price raised to 320 rubles.
A key risk remains potential rate hikes by the Central Bank.
Inflation expectations are at 7.5% for the next 12 months.
The bank plans to expand its AI-driven services in the next fiscal year.
Customer satisfaction scores increased by 5 points year over year.
Operational efficiency improved with cost-to-income ratio declining to 28.5%.
"""

summary = extractive_summarize(long_report, n_sentences=4)
print(f"Original: {len(long_report)} chars")
print(f"Summary:  {len(summary)} chars ({len(summary)/len(long_report)*100:.0f}%)")
print()
print("Summary:")
print(summary)

In [ ]:
# === Суммаризация с использованием предобученной модели (если доступна) ===
if TRANSFORMERS_AVAILABLE:
    try:
        from transformers import pipeline
        
        summarizer = pipeline(
            'summarization', 
            model='sshleifer/distilbart-cnn-12-6',
        )
        
        result = summarizer(long_report, max_length=100, min_length=30, do_sample=False)
        print("\nBERT-based Summary:")
        print(result[0]['summary_text'])
    except Exception as e:
        print(f"Summarization model not available: {e}")

## 📋 Выводы

1. **DistilBERT** значительно превосходит TF-IDF baseline на финансовых новостях
2. **NER** позволяет извлекать суммы, компании и даты из текста
3. **Шаблонная генерация** отчётов — надёжный подход для персонализированных финансовых рекомендаций
4. **Суммаризация** помогает обрабатывать длинные финансовые документы
5. **Best Practices:** Начинать с простых методов, переходить к трансформерам при необходимости